# Video Game Commercial Success Prediction & Market Analysis

The goal of this project is to deliver a clear commercial narrative that game studios and publishers can use to minimize financial risk before greenlighting a new game's production.

When looking into creating new games studios and publishers investigate what has been historically successful. By analyzing characteristics of games that have already been released, we can help guide new game ideas to the right publishers to help propagate the game to a better sales pattern, allowing both the publisher and game developer to maximize their investment.

#### Imports

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

## Step 1: Environment Setup & In-Memory Database Initialization (SQL)

To start, we will create an in-memory database using SQLAlchemy. 
By doing this, the data pipeline can be run anywhere.

In [ ]:
# Create an in memory SQLite database
engine = create_engine('sqlite:///:memory:')

In [ ]:
# Load data into pandas dataframe
df_raw = pd.read_csv('datasets/Video_Games_Sales.csv')
df_raw.info()

In [ ]:
# ingest raw data into the in-memory SQL database
df_raw.to_sql('raw_game_sales', con=engine, index=False, if_exists='replace')

In [ ]:
# Run SQL query directly against the database
query = """
SELECT count(*)
FROM raw_game_sales
"""

In [ ]:
with engine.connect() as conn:
    results = conn.execute(text(query))
    print(f'Successfully ingested {results.scalar()} records into in-memory table "raw_game_sales".')

## Step 2: Cleaning and Transformation (SQL)

Next we will pull the columns we need for the analysis. We will also filter out the `NULL` values from the `critic_scores`, `global_sales`, and `release_year`.

In [ ]:
# Query to pull only the needed columns from database
query = """
SELECT name,
    platform,
    year_of_release as release_year,
    genre, 
    publisher,
    na_sales,
    eu_sales,
    jp_sales,
    other_sales,
    global_sales,
    critic_score,
    user_score
FROM raw_game_sales
WHERE critic_score IS NOT NULL 
AND global_sales IS NOT NULL
AND year_of_release IS NOT NULL
AND user_score != 'tbd'
AND publisher IS NOT NULL;
"""

In [ ]:
with engine.connect() as conn:
    results = conn.execute(text(query))
    df_cleaned = pd.DataFrame(results)

In [ ]:
df_cleaned.info()

## Step 3: Exploratory Data Analysis & Feature Engineering (Python)

In [ ]:
# Change column to numeric
df_cleaned['User_Score'] = pd.to_numeric(df_cleaned['User_Score'], errors='coerce')

In [ ]:
# Remove rows containing NaN values
df_cleaned = df_cleaned.dropna()

In [ ]:
# Confirm change
df_cleaned.info()